# Whisper Transcription

تحويل الصوت إلى نص بدقة عالية باستخدام Whisper large-v3 على Kaggle GPU.

المخرجات:
- `transcript.json` (نص كامل + كلمات بمستوى الطابع الزمني word-level)
- `transcript.srt` (ترجمة SRT قابلة للتحرير)
- رفع النتائج إلى GitHub Release

In [ ]:
# Cell 2 — تثبيت Whisper + ffmpeg
import subprocess, sys
!pip install -q openai-whisper torch torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q ffmpeg-python
!apt-get update -qq && apt-get install -y -qq ffmpeg
import whisper, torch
print(f'Whisper version : {whisper.__version__ if hasattr(whisper, "__version__") else "unknown"}')
print(f'CUDA available   : {torch.cuda.is_available()}')
print(f'GPU              : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"}')

In [ ]:
# Cell 3 — تحميل النموذج (large-v3 للعربية)
import os, time
MODEL_NAME = os.environ.get('WHISPER_MODEL', 'large-v3')
print(f'Loading Whisper {MODEL_NAME} ... (this downloads ~3GB on first run)')
t0 = time.time()
model = whisper.load_model(MODEL_NAME)
if torch.cuda.is_available():
    model = model.cuda()
print(f'[OK] Model loaded in {time.time()-t0:.1f}s')

In [ ]:
# Cell 4 — تفريغ الفيديو/الصوت إلى نص
import os, json, whisper

input_path = os.environ.get('INPUT_MEDIA', '/kaggle/input/test_video.mp4')
if not os.path.exists(input_path):
    raise FileNotFoundError(f'Input media not found: {input_path}')

language = os.environ.get('WHISPER_LANGUAGE', 'ar')  # Arabic by default
task     = os.environ.get('WHISPER_TASK', 'transcribe')   # or 'translate'

print(f'Transcribing: {input_path}')
print(f'Language   : {language}')
print(f'Task       : {task}')

result = model.transcribe(
    input_path,
    language=language,
    task=task,
    word_timestamps=True,        # P4-T2: word-level timestamps
    verbose=False,
)
print(f'[OK] Transcribed {len(result["segments"])} segments, {len(result["text"])} chars')

In [ ]:
# Cell 5 — حفظ النتائج بصيغة JSON + SRT
import json, os

out_dir = '/kaggle/working/output'
os.makedirs(out_dir, exist_ok=True)

# --- JSON: full transcript + word-level timestamps (Hormozi caption generator input) ---
out_json = os.path.join(out_dir, 'transcript.json')
payload = {
    'text': result.get('text', ''),
    'language': result.get('language', language),
    'segments': [
        {
            'id': s['id'],
            'start': s['start'],
            'end':   s['end'],
            'text':  s['text'].strip(),
            'words': [
                {'word': w['word'], 'start': w['start'], 'end': w['end'], 'probability': w.get('probability', 1.0)}
                for w in s.get('words', [])
            ],
        }
        for s in result.get('segments', [])
    ],
}
with open(out_json, 'w', encoding='utf-8') as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)
print(f'[OK] Wrote {out_json}  ({os.path.getsize(out_json)/1024:.1f} KB)')

# --- SRT: standard subtitle format ---
def fmt(t: float) -> str:
    h = int(t // 3600); m = int((t % 3600) // 60); s = int(t % 60)
    ms = int((t - int(t)) * 1000)
    return f'{h:02d}:{m:02d}:{s:02d},{ms:03d}'

out_srt = os.path.join(out_dir, 'transcript.srt')
with open(out_srt, 'w', encoding='utf-8') as f:
    for i, s in enumerate(result.get('segments', []), 1):
        f.write(f'{i}\n{fmt(s["start"])} --> {fmt(s["end"])}\n{s["text"].strip()}\n\n')
print(f'[OK] Wrote {out_srt}  ({os.path.getsize(out_srt)/1024:.1f} KB)')

In [ ]:
# Cell 6 — رفع النتائج إلى GitHub Release
import os, glob
outputs = [
    '/kaggle/working/output/transcript.json',
    '/kaggle/working/output/transcript.srt',
]
outputs = [p for p in outputs if os.path.exists(p)]
print(f'Files to upload: {len(outputs)}')
for p in outputs:
    print(f'  - {p}  ({os.path.getsize(p)/1024:.2f} KB)')

import os
import json
import subprocess
import urllib.request
import urllib.error

# --- إعدادات الرفع إلى GitHub Release ---
# يتم تمريرها عبر Kaggle Secrets أو متغيرات البيئة
GH_TOKEN = os.environ.get('GH_TOKEN', os.environ.get('GITHUB_TOKEN', ''))
GH_REPO = os.environ.get('GH_REPO', 'mickystudio/micky-infra')
TASK_ID = os.environ.get('KAGGLE_TASK_ID', 'test')
RELEASE_TAG = os.environ.get('RELEASE_TAG', f'task-{TASK_ID}')


def gh_release_api(method: str, path: str, payload: dict | None = None,
                   upload_url: str | None = None, asset_path: str | None = None):
    """Calls the GitHub REST API (or uploads an asset) with auth."""
    headers = {
        'Accept': 'application/vnd.github+json',
        'Authorization': f'Bearer {GH_TOKEN}',
        'X-GitHub-Api-Version': '2022-11-28',
    }
    try:
        if upload_url and asset_path:
            # Upload asset via multipart form
            import mimetypes
            mime = mimetypes.guess_type(asset_path)[0] or 'application/octet-stream'
            headers['Content-Type'] = mime
            with open(asset_path, 'rb') as fh:
                req = urllib.request.Request(upload_url, data=fh.read(), headers=headers, method='POST')
        else:
            data = json.dumps(payload).encode() if payload else None
            base = 'https://api.github.com'
            url = f'{base}{path}' if upload_url is None else upload_url
            req = urllib.request.Request(url, data=data, headers=headers, method=method)
        with urllib.request.urlopen(req, timeout=120) as resp:
            body = resp.read().decode('utf-8', errors='ignore')
            return resp.status, json.loads(body) if body else {}
    except urllib.error.HTTPError as e:
        err_body = e.read().decode('utf-8', errors='ignore')
        print(f'GitHub API error {e.code}: {err_body}')
        return e.code, {}
    except Exception as e:
        print(f'GitHub upload exception: {e}')
        return 0, {}


def ensure_release():
    """Create the release if it doesn't exist, return its upload_url template."""
    # Try fetching first
    status, data = gh_release_api('GET', f'/repos/{GH_REPO}/releases/tags/{RELEASE_TAG}')
    if status == 200:
        upload_url = data.get('upload_url', '').replace('{?name,label}', '')
        return upload_url
    # Create the release
    payload = {
        'tag_name': RELEASE_TAG,
        'name': f'Kaggle task {TASK_ID}',
        'body': 'Auto-uploaded from Kaggle notebook',
        'draft': False,
        'prerelease': False,
    }
    status, data = gh_release_api('POST', f'/repos/{GH_REPO}/releases', payload=payload)
    return data.get('upload_url', '').replace('{?name,label}', '') if data else ''


def upload_asset(upload_url: str, asset_path: str) -> str | None:
    """Uploads a single file to the release. Returns the browser_download_url."""
    name = os.path.basename(asset_path)
    url = f'{upload_url}?name={name}'
    status, data = gh_release_api('POST', '', upload_url=url, asset_path=asset_path)
    if status in (201, 200) and data:
        print(f'  uploaded: {name} -> {data.get("browser_download_url")}')
        return data.get('browser_download_url')
    print(f'  FAILED to upload {name} (status={status})')
    return None


def upload_files_to_release(file_paths, manifest=None):
    """Uploads all files to the GitHub release for this task."""
    if not GH_TOKEN:
        print('WARNING: GH_TOKEN not set. Skipping upload to GitHub Release.')
        print('Files remain available on the Kaggle output at /kaggle/working/output')
        return []
    upload_url = ensure_release()
    if not upload_url:
        print('ERROR: could not create/find GitHub release. Aborting upload.')
        return []
    uploaded = []
    for fp in file_paths:
        if not os.path.exists(fp):
            print(f'  skip (missing): {fp}')
            continue
        url = upload_asset(upload_url, fp)
        if url:
            uploaded.append(url)
    # Optional manifest (JSON describing outputs)
    if manifest:
        manifest_path = '/kaggle/working/manifest.json'
        with open(manifest_path, 'w') as f:
            json.dump(manifest, f, ensure_ascii=False, indent=2)
        upload_asset(upload_url, manifest_path)
    return uploaded


print(f'Release tag: {RELEASE_TAG}')
print(f'Repo:         {GH_REPO}')
print(f'Token set:    {bool(GH_TOKEN)}')

manifest = {
    'task_id': os.environ.get('KAGGLE_TASK_ID', 'test'),
    'input': input_path,
    'language': language,
    'task': task,
    'segments': len(result.get('segments', [])),
    'outputs': [{'file': os.path.basename(p), 'size_kb': round(os.path.getsize(p)/1024, 2)} for p in outputs],
}
uploaded_urls = upload_files_to_release(outputs, manifest=manifest)
print(f'\n[OK] {len(uploaded_urls)} files uploaded to GitHub Release.')
for u in uploaded_urls:
    print(f'  {u}')

## تم!

ملف `transcript.json` جاهز ليُغذّى إلى `caption_generator.py` (P4-T2) لتوليد ترجمات بنمط Hormozi.